### 1. Imports 

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

### 2. Load Data

In [ ]:
BASE_PATH = "/kaggle/input/datasets/mouhamed221/dataset-week8/dataset/"

TRAIN_IMG = BASE_PATH + "train/images/"
TRAIN_MASK = BASE_PATH + "train/masks/"

VAL_IMG = BASE_PATH + "val/images/"
VAL_MASK = BASE_PATH + "val/masks/"

TEST_IMG = BASE_PATH + "test/images/"

class BrainDataset(Dataset):
    def __init__(self, img_dir, mask_dir=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(img_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]

        img_path = os.path.join(self.img_dir, img_name)
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
        if image is None:
            raise FileNotFoundError(f"Image not found: {img_path}")
            
        image = cv2.resize(image, (256, 256))
        image = image / 255.0
        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)

        if self.mask_dir:
            base = os.path.splitext(img_name)[0]
            mask_name = base + "_mask.png"
            mask_path = os.path.join(self.mask_dir, mask_name)
        
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
            if mask is None:
                raise FileNotFoundError(f"Mask not found: {mask_path}")
        
            mask = cv2.resize(mask, (256, 256))
            mask = mask / 255.0
            mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)

            return image, mask
        else:
            return image, img_name


train_dataset = BrainDataset(TRAIN_IMG, TRAIN_MASK)
val_dataset = BrainDataset(VAL_IMG, VAL_MASK)
test_dataset = BrainDataset(TEST_IMG)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

### 3. Modèle U-Net

In [ ]:
class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.ReLU(),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.ReLU()
            )

        self.enc1 = conv_block(1, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = conv_block(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = conv_block(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = conv_block(256, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = conv_block(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = conv_block(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = conv_block(128, 64)

        self.final = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))

        b = self.bottleneck(self.pool3(e3))

        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.final(d1)

### 3. Loss + Device

In [ ]:
def dice_coef(pred, target, smooth=1e-6):
    pred = torch.sigmoid(pred)
    pred = pred.view(-1)
    target = target.view(-1)

    intersection = (pred * target).sum()
    return (2 * intersection + smooth) / (pred.sum() + target.sum() + smooth)

def dice_loss(pred, target):
    return 1 - dice_coef(pred, target)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

### 3. Training

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    train_loss = 0

    for images, masks in tqdm(train_loader):
        images, masks = images.to(device), masks.to(device)

        outputs = model(images)
        loss = dice_loss(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Validation
    model.eval()
    val_dice = 0

    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)

            outputs = model(images)
            val_dice += dice_coef(outputs, masks).item()

    val_dice /= len(val_loader)

    print(f"Epoch {epoch+1} | Loss: {train_loss:.4f} | Dice: {val_dice:.4f}")

### 3. Prédictions

In [ ]:
predictions = {}

model.eval()
with torch.no_grad():
    for images, image_ids in test_loader:
        images = images.to(device)

        outputs = model(images)
        outputs = torch.sigmoid(outputs).cpu().numpy()

        for pred, img_id in zip(outputs, image_ids):
            predictions[img_id] = pred.squeeze()

### 3. RLE Encoding and file Submission

In [ ]:
def mask_to_rle(mask, threshold=0.5):
    binary = (mask > threshold).astype(np.uint8)

    if binary.sum() == 0:
        return ""

    pixels = binary.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])

    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]

    return ' '.join(str(x) for x in runs)
    

rows = []

for image_id, mask in predictions.items():
    rle = mask_to_rle(mask)

    rows.append({
        "image_id": image_id,
        "rle_mask": rle
    })

submission = pd.DataFrame(rows)
submission.to_csv("/kaggle/working/submission.csv", index=False)

print("✅ submission.csv is OK :) ")